# 02b. MLflow Tracking - Experiment Logging

Трекинг экспериментов в MLflow/DagsHub.

**Трекинг:** MLflow/DagsHub
**Модели:** Ridge, RandomForest, LightGBM, CatBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

## MLflow Configuration

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "https://dagshub.com/Zerol-91/Stress-Level-ML-Researching.mlflow"

try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("Anxiety_Prediction")
    print("MLflow connected successfully")
    MLFLOW_ACTIVE = True
except Exception as e:
    print(f"MLflow connection failed: {e}")
    print("Continuing without MLflow tracking...")
    MLFLOW_ACTIVE = False

## Load Data

In [ ]:
df = pd.read_csv('../data/processed_data.csv')

TARGET = 'Anxiety_Level'
FEATURES = [c for c in df.columns if c not in [TARGET, 'Subjective_Anxiety']]

X = df[FEATURES]
y = df[TARGET]

print(f"Features: {len(FEATURES)}, Samples: {len(y)}")
print(f"Target range: {y.min()} - {y.max()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=pd.qcut(y, q=5, labels=False, duplicates='drop')
)

y_strat = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, name=''):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    n, p = len(y_test), X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    print(f"\n{name}")
    print(f"  MAE: {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  R²: {r2:.3f}")
    print(f"  Adj R²: {adj_r2:.3f}")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'adj_r2': adj_r2, 'predictions': y_pred, 'model': model}

In [ ]:
def train_bagging_cv(model_class, params, X_train, y_train, n_splits=5):
    models = []
    predictions = []
    
    for train_idx, val_idx in stratified_cv.split(X_train, y_strat):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = model_class(**params)
        model.fit(X_tr, y_tr)
        models.append(model)
        predictions.append(model.predict(X_val))
    
    return models, np.mean(predictions, axis=0)

## 1. Ridge Regression Baseline

In [ ]:
with mlflow.start_run(run_name="Ridge_Baseline") as run:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    ridge = Ridge(alpha=1.0)
    ridge_results = evaluate_model(ridge, X_train_scaled, X_test_scaled, y_train, y_test, 'Ridge Regression')
    
    mlflow.log_params({'alpha': 1.0, 'scaler': 'StandardScaler'})
    mlflow.log_metrics({
        'MAE': ridge_results['mae'],
        'RMSE': ridge_results['rmse'],
        'R2': ridge_results['r2'],
        'Adj_R2': ridge_results['adj_r2']
    })
    print(f"\nRun ID: {run.info.run_id}")

## 2. Random Forest with Optuna

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestRegressor(**params)
    
    train_scores, val_scores = [], []
    for train_idx, val_idx in stratified_cv.split(X_train, y_strat):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        train_scores.append(mean_absolute_error(y_tr, model.predict(X_tr)))
        val_scores.append(mean_absolute_error(y_val, model.predict(X_val)))
    
    train_mae = np.mean(train_scores)
    val_mae = np.mean(val_scores)
    gap_penalty = max(0, val_mae - train_mae) * 0.5
    
    return val_mae + gap_penalty

study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f"\nBest MAE: {study_rf.best_value:.3f}")
print(f"Best params: {study_rf.best_params}")

In [ ]:
with mlflow.start_run(run_name="RF_Optuna") as run:
    rf_optuna = RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1)
    rf_results = evaluate_model(rf_optuna, X_train, X_test, y_train, y_test, 'Random Forest + Optuna')
    
    mlflow.log_params(study_rf.best_params)
    mlflow.log_metrics({
        'MAE': rf_results['mae'],
        'RMSE': rf_results['rmse'],
        'R2': rf_results['r2'],
        'Adj_R2': rf_results['adj_r2']
    })
    print(f"\nRun ID: {run.info.run_id}")

## 3. LightGBM with Optuna

In [ ]:
def objective_lgb(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 8, 64),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)

study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

print(f"\nBest MAE: {study_lgb.best_value:.3f}")

In [ ]:
with mlflow.start_run(run_name="LightGBM_Optuna") as run:
    lgb_model = lgb.LGBMRegressor(**study_lgb.best_params, verbosity=-1, random_state=42)
    lgb_results = evaluate_model(lgb_model, X_train, X_test, y_train, y_test, 'LightGBM + Optuna')
    
    mlflow.log_params(study_lgb.best_params)
    mlflow.log_metrics({
        'MAE': lgb_results['mae'],
        'RMSE': lgb_results['rmse'],
        'R2': lgb_results['r2'],
        'Adj_R2': lgb_results['adj_r2']
    })
    print(f"\nRun ID: {run.info.run_id}")

## 4. LightGBM: Bagging Ensemble

In [ ]:
lgb_params = {**study_lgb.best_params, 'verbosity': -1, 'random_state': 42}
lgb_models, lgb_cv_preds = train_bagging_cv(lgb.LGBMRegressor, lgb_params, X_train, y_train)

test_preds = np.array([m.predict(X_test) for m in lgb_models])
lgb_bagging_preds = np.mean(test_preds, axis=0)

mae = mean_absolute_error(y_test, lgb_bagging_preds)
rmse = np.sqrt(mean_squared_error(y_test, lgb_bagging_preds))
r2 = r2_score(y_test, lgb_bagging_preds)

print(f"\nLightGBM Bagging Ensemble")
print(f"  MAE: {mae:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"  R²: {r2:.3f}")

In [ ]:
with mlflow.start_run(run_name="LightGBM_Bagging") as run:
    mlflow.log_params({'n_models': len(lgb_models), 'strategy': 'bagging', **lgb_params})
    mlflow.log_metrics({
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })
    print(f"Run ID: {run.info.run_id}")

## 5. CatBoost with Optuna

In [ ]:
def objective_cat(trial):
    params = {
        'loss_function': 'MAE',
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_seed': 42,
        'verbose': 0
    }
    
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=30, show_progress_bar=True)

print(f"\nBest MAE: {study_cat.best_value:.3f}")

In [ ]:
with mlflow.start_run(run_name="CatBoost_Optuna") as run:
    cat_model = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=0)
    cat_results = evaluate_model(cat_model, X_train, X_test, y_train, y_test, 'CatBoost + Optuna')
    
    mlflow.log_params(study_cat.best_params)
    mlflow.log_metrics({
        'MAE': cat_results['mae'],
        'RMSE': cat_results['rmse'],
        'R2': cat_results['r2'],
        'Adj_R2': cat_results['adj_r2']
    })
    print(f"\nRun ID: {run.info.run_id}")

## 6. CatBoost: Bagging Ensemble

In [ ]:
cat_params = {**study_cat.best_params, 'random_seed': 42, 'verbose': 0}
cat_models, cat_cv_preds = train_bagging_cv(CatBoostRegressor, cat_params, X_train, y_train)

test_preds = np.array([m.predict(X_test) for m in cat_models])
cat_bagging_preds = np.mean(test_preds, axis=0)

mae = mean_absolute_error(y_test, cat_bagging_preds)
rmse = np.sqrt(mean_squared_error(y_test, cat_bagging_preds))
r2 = r2_score(y_test, cat_bagging_preds)

print(f"\nCatBoost Bagging Ensemble")
print(f"  MAE: {mae:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"  R²: {r2:.3f}")

In [ ]:
with mlflow.start_run(run_name="CatBoost_Bagging") as run:
    mlflow.log_params({'n_models': len(cat_models), 'strategy': 'bagging', **cat_params})
    mlflow.log_metrics({
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })
    print(f"Run ID: {run.info.run_id}")

## View Experiments in MLflow UI

In [ ]:
client = MlflowClient()
experiment = client.get_experiment_by_name("Anxiety_Prediction")

runs = client.search_runs([experiment.experiment_id], order_by=["metrics.MAE ASC"])

print(f"\nExperiment: {experiment.name}")
print(f"Total runs: {len(runs)}")
print("\nBest runs by MAE:")
for run in runs[:5]:
    print(f"  {run.data.tags.get('mlflow.runName', 'N/A')}: MAE={run.data.metrics.get('MAE', 0):.3f}, R2={run.data.metrics.get('R2', 0):.3f}")